# Session 8 — Threat Intelligence Extraction and Summarisation

**ISEA · Intelligent Digital Forensics: LLMs, NLP, and Explainable AI for Cyber Investigations**

Two solo labs, thirty minutes each.

| | Lab | Sections | What you produce |
|---|---|---|---|
| **Lab 1** | Extract by rule | 1 – 4 | An extraction, a filled pyramid, three prioritisation answers |
| **Lab 2** | Summarise by model, then audit | 5 – 6 | An LLM summary and a fabrication report |

**Runtime: CPU.** The one rule this session exists for:

> **Extract by rule. Summarise by model. Never the other way round.**

Everything below runs against **live public data** — a real CISA advisory, the NVD API, CISA's
known-exploited catalogue and the current EPSS scores. Your numbers will be close to the deck's
but **not identical**, because these feeds move daily. That is the point, and you should say so
in anything you write from them.

---
## Section 0 — Setup

Run this first. The notebook now uses an **offline-first fallback** for the two CISA sources:

1. If `advisory.txt` or `known_exploited_vulnerabilities.json` is already present, it uses the local copy.
2. Otherwise it tries the live CISA source.
3. If CISA returns `403 Forbidden` or another network error in Colab, a file-upload chooser opens automatically.

The NVD and EPSS calls used later in the notebook remain live, so their figures can still move daily.


In [ ]:
import re, json, gzip, io, time, html, urllib.request, os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Browser-like headers reduce avoidable blocks, but CISA may still reject
# automated requests from Colab. The notebook therefore has a local/upload fallback.
UA = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
}

def get(url, timeout=120):
    req = urllib.request.Request(url, headers=UA)
    return urllib.request.urlopen(req, timeout=timeout).read()

def colab_upload(expected_name, description):
    """Open Colab's upload chooser and return uploaded bytes."""
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError(
            f"\nLive download failed and {expected_name!r} is not present.\n"
            f"Place the supplied {expected_name} beside this notebook and run the cell again."
        )

    print(f"\nPlease upload the supplied {expected_name} ({description}).")
    uploaded = files.upload()

    if expected_name in uploaded:
        data = uploaded[expected_name]
    elif len(uploaded) == 1:
        # Accept a single uploaded file even if the browser renamed it.
        data = next(iter(uploaded.values()))
    else:
        raise RuntimeError(
            f"Could not identify {expected_name}. Upload that file only, then run the cell again."
        )

    with open(expected_name, "wb") as f:
        f.write(data)
    return data

# ---------------------------------------------------------------------
# 1. CISA joint advisory — LOCAL FIRST, LIVE SECOND, UPLOAD THIRD
# ---------------------------------------------------------------------
ADV = "aa26-204a"
ADVISORY_FILE = "advisory.txt"
ADVISORY_URL = f"https://www.cisa.gov/news-events/cybersecurity-advisories/{ADV}"

if os.path.exists(ADVISORY_FILE) and os.path.getsize(ADVISORY_FILE) > 1000:
    TEXT = open(ADVISORY_FILE, encoding="utf-8", errors="replace").read()
    print("advisory : using local advisory.txt")
else:
    try:
        raw = get(ADVISORY_URL).decode("utf-8", "replace")
        t = re.sub(r"<script.*?</script>|<style.*?</style>", "", raw, flags=re.S)
        TEXT = re.sub(r"[ \t]+", " ", html.unescape(re.sub(r"<[^>]+>", " ", t)))
        with open(ADVISORY_FILE, "w", encoding="utf-8") as f:
            f.write(TEXT)
        print("advisory : downloaded live from CISA")
    except Exception as e:
        print(f"advisory : live CISA fetch failed ({type(e).__name__}: {e})")
        data = colab_upload(ADVISORY_FILE, "offline copy of CISA advisory AA26-204A")
        TEXT = data.decode("utf-8", "replace")
        print("advisory : using uploaded advisory.txt")

# ---------------------------------------------------------------------
# 2. CISA Known Exploited Vulnerabilities — same fallback pattern
# ---------------------------------------------------------------------
KEV_FILE = "known_exploited_vulnerabilities.json"
KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

if os.path.exists(KEV_FILE) and os.path.getsize(KEV_FILE) > 1000:
    with open(KEV_FILE, encoding="utf-8") as f:
        KEV = json.load(f)
    print("KEV      : using local known_exploited_vulnerabilities.json")
else:
    try:
        kev_blob = get(KEV_URL)
        KEV = json.loads(kev_blob)
        with open(KEV_FILE, "wb") as f:
            f.write(kev_blob)
        print("KEV      : downloaded live from CISA")
    except Exception as e:
        print(f"KEV      : live CISA fetch failed ({type(e).__name__}: {e})")
        data = colab_upload(KEV_FILE, "offline copy of the CISA KEV catalogue")
        KEV = json.loads(data.decode("utf-8", "replace"))
        print("KEV      : using uploaded KEV file")

kev_ids = {v["cveID"] for v in KEV["vulnerabilities"]}

print()
print(f"advisory : {len(TEXT):,} characters")
print(f"KEV      : {KEV['count']:,} entries, released {KEV['dateReleased'][:10]}")


> **Built-in fallback.** If CISA blocks Colab with `HTTP 403`, do **not** edit the code. The cell will automatically ask you to upload `advisory.txt` (and, if needed, `known_exploited_vulnerabilities.json`). The facilitator should keep both offline files available. Once uploaded, rerun Section 0 and the notebook will use the local copies first on subsequent runs.


---
## Section 1 — Extract by rule  ·  **LAB 1, Part A**

Everything that will be *actioned* gets extracted deterministically. A regex either matches or it
does not: auditable, reproducible, free, and structurally incapable of inventing an indicator that
was not in the source.

In [ ]:
IPV4   = r"(?<![\w.])(?:(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)\.){3}(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)(?![\w.])"
TLD    = r"(?:com|net|org|ru|ua|nl|io|info|biz|co|uk|de|fr|cn|top|xyz|online|site|shop|app|dev|cloud|email|link)"
DOM    = rf"(?<![\w.@-])(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{{0,61}}[a-zA-Z0-9])?\.)+{TLD}(?![\w-])"
SHA256 = r"(?<![a-fA-F0-9])[a-fA-F0-9]{64}(?![a-fA-F0-9])"
MD5    = r"(?<![a-fA-F0-9])[a-fA-F0-9]{32}(?![a-fA-F0-9])"
CVE    = r"CVE-\d{4}-\d{4,7}"
ATTACK = r"\bT1\d{3}(?:\.\d{3})?\b"

def extract(text):
    return {"ipv4":   set(re.findall(IPV4, text)),
            "domain": set(d.lower() for d in re.findall(DOM, text)),
            "sha256": set(h.lower() for h in re.findall(SHA256, text)),
            "md5":    set(h.lower() for h in re.findall(MD5, text)),
            "cve":    set(re.findall(CVE, text)),
            "attack": set(re.findall(ATTACK, text))}

naive = extract(TEXT)
for k, v in naive.items():
    print(f"{k:8} {len(v)}")

Look at that IP count before you go on.

An advisory that describes adversary infrastructure across dozens of paragraphs reported **zero**
IP addresses. The regex is correct. The assumption behind it is not.

In [ ]:
# what does the document actually look like?
for m in re.finditer(r"\S*\[\.\]\S*", TEXT):
    print(m.group(0))
    if m.start() > 40000: break

### Refanging

Advisories **defang** indicators on purpose — so a reader cannot click a live malicious link by
accident, and so the document does not trip a mail filter. `1.2.3[.]4` is not an IPv4 address to
a pattern matcher; it is a string with brackets in it.

The fix is one substitution, applied *before* extraction.

In [ ]:
def refang(text):
    text = re.sub(r"\[\.\]|\(\.\)|\{\.\}", ".", text)
    text = re.sub(r"\[:\]", ":", text)
    text = re.sub(r"\bh(?:xx|XX)(p|ps)\b", r"htt\1", text, flags=re.I)
    text = re.sub(r"\[(?:at|@)\]", "@", text, flags=re.I)
    return text

aware = extract(refang(TEXT))

cmp = pd.DataFrame({"naive": {k: len(v) for k, v in naive.items()},
                    "refanged": {k: len(v) for k, v in aware.items()}})
cmp["missed"] = cmp.refanged - cmp.naive
print(cmp.to_string())

net_n = len(naive["ipv4"]) + len(naive["domain"])
net_a = len(aware["ipv4"]) + len(aware["domain"])
print(f"\nnetwork indicators: {net_n} of {net_a}")
print(f"naive recall = {100*net_n/net_a:.1f}%")

**Expected, on the advisory as fetched for the deck:**

```
        naive  refanged  missed
ipv4        0         9       9
domain     10        19       9
sha256      5         5       0
md5         0         0       0
cve         1         1       0
attack     36        36       0

network indicators: 10 of 28
naive recall = 35.7%
```

Hashes and technique IDs are unaffected — they contain no dots to defang. **The entire loss is in
the network indicators**, which is the half that gets pushed into blocklists.

A pipeline built the naive way does not error. It reports ten indicators, cleanly, every day. The
only way to notice is to read the source yourself — which is the reading the automation was bought
to save you.

In [ ]:
print("Recovered only after refanging:\n")
for kind in ("ipv4", "domain"):
    for x in sorted(aware[kind] - naive[kind]):
        print(f"  {kind:7} {x}")

---
## Section 2 — Fill the pyramid  ·  **LAB 1, Part B**

Bianco's Pyramid of Pain (2013). The vertical axis is not importance — it is **how much it costs
the adversary to change**. A hash dies when one byte changes. Changing how you work is expensive,
slow and visible.

In [ ]:
TIERS = [("TTPs (ATT&CK techniques)", len(aware["attack"])),
         ("Tools",                    0),
         ("Network / host artefacts", 0),
         ("Domain names",             len(aware["domain"])),
         ("IP addresses",             len(aware["ipv4"])),
         ("Hash values",              len(aware["sha256"]) + len(aware["md5"]))]

bottom = sum(c for n, c in TIERS[3:])
top    = sum(c for n, c in TIERS[:3])
for n, c in TIERS:
    print(f"{n:28} {c:3}")
print(f"\nbelow the line (cheap to change): {bottom}")
print(f"above the line (expensive)      : {top}")
print(f"share below the line: {100*bottom/(bottom+top):.1f}%")

**Two things to write down.**

1. This advisory is **unusually good**: roughly half its extractable value sits above the bottom
   three tiers. Most commercial indicator feeds are 100% bottom three tiers, because that is the
   part that is cheap to produce at volume.
2. A feed priced per indicator is priced by the tier that expires fastest. That is not an accident
   and it is worth saying in a procurement conversation.

---
## Section 3 — Severity is not exploitation is not risk  ·  **LAB 1, Part C**

Three signals that answer three different questions and get collapsed into one number constantly.

In [ ]:
def nvd_count(q):
    u = f"https://services.nvd.nist.gov/rest/json/cves/2.0?{q}&resultsPerPage=1"
    return json.loads(get(u, timeout=90))["totalResults"]

critical      = nvd_count("cvssV3Severity=CRITICAL");     time.sleep(7)
critical_kev  = nvd_count("cvssV3Severity=CRITICAL&hasKev"); time.sleep(7)
kev_total     = nvd_count("hasKev")

print(f"CVEs rated CRITICAL            : {critical:,}")
print(f"...of which known exploited    : {critical_kev:,}  ({100*critical_kev/critical:.2f}%)")
print(f"Known exploited, total         : {kev_total:,}")
print(f"...NOT rated critical          : {kev_total-critical_kev:,}  ({100*(kev_total-critical_kev)/kev_total:.1f}%)")

Patch strictly by CVSS and you work through **thirty thousand** vulnerabilities to reach the few
hundred anyone is actually exploiting — and you still miss the ones below your threshold that are
being used against people right now.

And none of these is *your* risk. Risk depends on whether you run the software, whether it is
exposed, and what it protects. A CVSS 10 in a product you do not deploy is a zero for you.

In [ ]:
# EPSS: prediction, versus KEV: observation
blob = gzip.decompress(get("https://epss.empiricalsecurity.com/epss_scores-current.csv.gz", timeout=180))
epss = pd.read_csv(io.StringIO("\n".join(blob.decode().split("\n")[1:])))
epss.columns = [c.strip() for c in epss.columns]
epss["is_kev"] = epss.cve.isin(kev_ids)

topN = epss.nlargest(kev_total, "epss")
print(f"EPSS scores {len(epss):,} CVEs")
print(f"top {kev_total:,} by EPSS contains {topN.is_kev.sum():,} known-exploited "
      f"({100*topN.is_kev.sum()/kev_total:.1f}%)")
print(f"median EPSS percentile of a known-exploited CVE: {100*epss[epss.is_kev].percentile.median():.1f}")

Be fair to EPSS. Ranking known-exploited CVEs at a median percentile near 99 is genuinely good,
and far better than CVSS. But it remains a **prediction**. Only the KEV catalogue is a record of
something that happened — and even that is a floor: it lists what was observed and reported, not
everything that occurred. **Absence from KEV is not evidence of absence of exploitation.**

**Write in your worksheet:** pick one CVE from the advisory. Get its severity, its KEV status and
its EPSS score. Then one sentence — which of the three would you actually brief, and to whom?

In [ ]:
cve = sorted(aware["cve"])[0] if aware["cve"] else "CVE-2021-44228"
row = epss[epss.cve == cve]
print(cve)
print(f"  in KEV : {cve in kev_ids}")
if len(row):
    print(f"  EPSS   : {row.epss.iloc[0]:.5f}   percentile {100*row.percentile.iloc[0]:.1f}")
else:
    print("  EPSS   : not scored")

---
## Section 4 — Build the audit harness  ·  **LAB 2 setup**

This is the whole defence, and it is four lines. Build it once, and put it between your model and
anything that writes to an enforcement system.

In [ ]:
TRUTH = extract(refang(TEXT))

def audit(summary_text):
    claimed = extract(refang(summary_text))
    report = {}
    for kind, vals in claimed.items():
        suspect = vals - TRUTH[kind]
        if vals or suspect:
            report[kind] = {"claimed": len(vals), "in_source": len(vals & TRUTH[kind]),
                            "NOT_IN_SOURCE": sorted(suspect)}
    return report

def show(report):
    bad = 0
    for kind, r in report.items():
        flag = "  <-- FABRICATED" if r["NOT_IN_SOURCE"] else ""
        print(f"{kind:8} claimed {r['claimed']:3}  traced {r['in_source']:3}{flag}")
        for x in r["NOT_IN_SOURCE"]:
            print(f"         !! {x}")
            bad += 1
    print(f"\n{bad} indicator(s) could not be traced to the source document.")
    return bad

print("harness ready")

### Prove it works before you trust it

Two test fixtures below. **I wrote both of these by hand** — they are not model output, and the
notebook does not pretend otherwise. The first is clean; the second contains one manufactured
hash and one manufactured CVE.

> **A warning worth more than the fixture.** The obvious way to build a fake CVE is to take a real
> one and add one to the last digit. Do not. When this notebook was written, doing exactly that to
> `CVE-2025-66376` produced `CVE-2025-66377` — **a real, unrelated vulnerability**. A hallucinated
> identifier does not always fail loudly: sometimes it resolves, and a team patches the wrong
> product and reports the work done. `CVE-2025-99999` is used below because it is genuinely
> unassigned. A 256-bit hash is different — the space is large enough that altering one character
> will not collide with anything.

In [ ]:
h_real   = sorted(TRUTH["sha256"])[0]
cve_real = sorted(TRUTH["cve"])[0]
h_fake   = h_real[:-1] + ("0" if h_real[-1] != "0" else "1")

# NOTE: do NOT build a fake CVE by incrementing a digit of a real one.
# We tried that while writing this notebook and it produced another REAL CVE,
# unrelated to this advisory. CVE-2025-99999 is unassigned (checked against the
# NVD API, zero results), so it is safe to use as a known-bad test string.
cve_fake = "CVE-2025-99999"

clean = f"The advisory describes a phishing campaign. Reported artefacts include {h_real} and {cve_real}."
dirty = (f"The advisory describes a phishing campaign. Reported artefacts include {h_real}, "
         f"{h_fake} and {cve_real}, alongside {cve_fake} affecting the same product.")

print("--- clean fixture ---");  show(audit(clean))
print("\n--- fixture with two manufactured indicators ---"); show(audit(dirty))

The harness catches both manufactured strings and passes the real ones. That is the guarantee:
**nothing appeared from nowhere.**

It is a narrow guarantee, and it is the one you need — indicators are the part that gets actioned
automatically, while prose is read by a human who can push back.

---
## Section 5 — Summarise by model  ·  **LAB 2, Part A**

Run the cell, paste the block into ChatGPT, and choose **one** audience.

In [ ]:
AUDIENCE = "a SOC engineer"        # or: "a board member with no technical background"

prompt = f'''
Write a short advisory summary for {AUDIENCE}, using ONLY the facts below.
Do not add indicators, CVE identifiers, actors, dates or product names that are not listed here.

SOURCE: CISA advisory {ADV.upper()}
- ATT&CK techniques cited: {len(TRUTH["attack"])}
- File hashes (SHA-256): {len(TRUTH["sha256"])}
- Domains: {len(TRUTH["domain"])}
- IP addresses: {len(TRUTH["ipv4"])}
- CVE identifiers: {", ".join(sorted(TRUTH["cve"])) or "none"}
- Example hashes: {", ".join(sorted(TRUTH["sha256"])[:2])}
- Example domains: {", ".join(sorted(TRUTH["domain"])[:3])}

Maximum 180 words. State a likelihood using one of: almost certainly / highly likely / likely /
roughly even chance / unlikely / highly unlikely. Separately state your confidence in the evidence.
'''
print(prompt)

---
## Section 6 — Audit it  ·  **LAB 2, Part B**

Paste the model's summary between the triple quotes and run.

In [ ]:
summary = '''
PASTE THE MODEL SUMMARY HERE
'''

if "PASTE THE MODEL" in summary:
    print("Paste a summary first, then re-run this cell.")
else:
    show(audit(summary))

### Now do the half the harness cannot

Set-difference catches **manufactured** indicators. It cannot catch a **real indicator described
wrongly** — if the summary calls an address adversary infrastructure when the advisory listed it
as a compromised victim, every string checks out and the meaning is inverted.

For each indicator the harness passed, open `advisory.txt` and check how the source describes it:

| Category | Question |
|---|---|
| Adversary infrastructure | Did the advisory say this was attacker-controlled? |
| Compromised third party | Or a victim whose systems were abused? |
| Example / documentation | Or an illustration the author invented? |
| Unstated | Did the advisory simply list it without saying? |

Blocking the wrong one breaks production. This is the professional work; the extraction was the
technical work.

**Write in your worksheet:** how many indicators the harness flagged, and how many it passed that
you still had to correct by reading.

---
## Section 7 — Worksheet  ·  **LAB 2, Part C**

In [ ]:
worksheet = {
  "name": "",
  "date": "",

  # ---- Lab 1
  "L1_naive_network_indicators":  None,
  "L1_refanged_network_indicators": None,
  "L1_naive_recall_pct":          None,
  "L1_pyramid_bottom_pct":        None,
  "L1_cve_examined":              "",
  "L1_which_signal_you_would_brief": "",

  # ---- Lab 2
  "L2_audience":                  "",
  "L2_llm_model":                 "",   # model + version + date
  "L2_indicators_flagged":        None, # manufactured, caught by the harness
  "L2_indicators_miscategorised": None, # real but described wrongly, caught by you

  # ---- the deliverable
  "CLAIM_rewritten": "",   # one sentence, with likelihood AND confidence, stated separately
  "SCALE_used": "",        # the likelihood scale you used, so a reader can check your meaning
}

missing = [k for k, v in worksheet.items() if v in ("", None)]
print(f"{len(worksheet)-len(missing)}/{len(worksheet)} fields complete")
if missing: print("still empty:", ", ".join(missing))

**A worked example of the rewritten claim:**

> *"The campaign described in AA26-204A is **highly likely** to be ongoing against similar targets.
> **Confidence: moderate** — the assessment rests on a single government advisory with no
> independent corroboration available to us, and we have not observed the described activity on
> our own estate. Likelihood scale: almost certainly (>90%) / highly likely (75–90%) / likely
> (55–75%) / roughly even chance (45–55%) / unlikely (25–45%) / highly unlikely (<25%)."*

Two separate axes. "Highly likely, moderate confidence" is a coherent and common position —
collapsing them into "we believe" destroys both, and 'possible' has been shown to mean anything
from five per cent to eighty per cent depending on who reads it.

---

## Sources

- Bianco, D. (2013). *The Pyramid of Pain.* https://detect-respond.blogspot.com/2013/03/the-pyramid-of-pain.html
- MITRE ATT&CK — enterprise tactics and techniques catalogue. https://attack.mitre.org/
- CISA, *Known Exploited Vulnerabilities Catalog.* https://www.cisa.gov/known-exploited-vulnerabilities-catalog
- FIRST, *Exploit Prediction Scoring System (EPSS).* https://www.first.org/epss/
- FIRST, *CVSS v3.1 specification.* https://www.first.org/cvss/v3.1/specification-document
- CISA advisory AA26-204A. https://www.cisa.gov/news-events/cybersecurity-advisories/aa26-204a
- NVD API endpoint used above: https://services.nvd.nist.gov/rest/json/cves/2.0 (developer docs at nvd.nist.gov/developers/vulnerabilities)
- Kent, S. *Words of Estimative Probability.* https://www.cia.gov/resources/csi/studies-in-intelligence/archives/vol-8-no-4/words-of-estimative-probability/